# Exercise 2: Keyword search with lemmas, prefixes and subwords

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Meaning and Retrieval · version 0.1.0*

> On KazQAD, compare BM25 over surface forms, prefixes (choose *k* on validation), analyser lemmas and subwords. Inspect the test queries where lemmas lose most: how many losses come from wrong or merging lemmas?

**Setup.** `pip install torch transformers sentencepiece scipy bm25s requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). Encoders are loaded at the Hugging Face revisions pinned in `MODELS`, from a local Hugging Face cache (`HF_HUB_DIR`), a folder of copies (`MODELS_DIR`) or the Hub; NLLB-200 is read from TurkicNLP's model folder (`~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M`) as the `embeddings` processor does. Data come from commit-pinned GitHub repositories: SIB-200 (FLORES-200 sentences, the same 1,004 sentences in every language), Kardeş-NLU's STS sets and KazQAD's retrieval collection (825,309 Kazakh Wikipedia passages, about 230 MB compressed). Embeddings, lemma maps and retrieval runs are cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`). Everything runs on the CPU.

KazQAD's retrieval collection has 825,309 passages from Kazakh Wikipedia, with relevance judgements for 548 validation and 1,929 test questions. Every passage and question is reduced in the same way before indexing. Lemmas come from TurkicNLP's Apertium analyser, applied once per word type without context (names are retried capitalised); subwords are Glot500's SentencePiece pieces. The first run builds the lemma map and seven indexes, about 15 minutes on two CPU threads.

In [ ]:
import collections, csv, gzip, hashlib, io, json, math, os, random, re, statistics, time
from pathlib import Path
import numpy as np, requests

SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KARDES = "https://raw.githubusercontent.com/lksenel/Kardes-NLU/48eea77fda904e731d228de833f4c57d130fad9c/Data/"
KAZQAD = "https://raw.githubusercontent.com/IS2AI/KazQAD/ab8684a667109f2a7345f4f89ab3b123e8d78e1f/data/information-retrieval/"
MODELS = {   # sentence encoder -> pinned revision (mean pooling, 128-token ceiling, as configured)
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "e8f8c211226b894fcb81acc59f3b34ba3efd5f42",
    "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "c2e19453aae46f98a865abfdf173f35f2e119f5a",
    "cis-lmu/glot500-base": "d4d7c1ec01828fdf7452a4ccf7b55177aced175e",                 # tokenizer only
    "meta-llama/Meta-Llama-3-8B": "8cde5ca8380496c9a6cc7ef3a8b46a0372a1d920",          # tokenizer only
    "allenai/Olmo-3-1025-7B": "0a15c289b17ad4f27aabc863276ec525d5312c01",              # tokenizer only
    "Qwen/Qwen3.6-27B": "6a9e13bd6fc8f0983b9b99948120bc37f49c13e9",                    # tokenizer only
}
NAMES = {"sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2": "MiniLM",
         "sentence-transformers/paraphrase-xlm-r-multilingual-v1": "XLM-R para", "nllb": "NLLB enc"}
CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
HF_HUB_DIR = Path(os.environ.get("HF_HUB_DIR", "~/.cache/huggingface/hub")).expanduser()
MODELS_DIR = Path(os.environ.get("MODELS_DIR", CACHE / "models"))
NLLB_DIR = Path("~/.turkicnlp/models/huggingface/facebook--nllb-200-distilled-600M").expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=600); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    FETCHED[url] = h.hexdigest()
    return path

def model_source(name):
    rev = MODELS[name]
    for p in (HF_HUB_DIR / f"models--{name.replace('/', '--')}" / "snapshots" / rev,
              MODELS_DIR / f"models--{name.replace('/', '--')}--{rev}"):
        if (p / "config.json").exists():
            return str(p), None
    return name, rev

def tokenizer(name):
    from transformers import AutoTokenizer
    path, rev = model_source(name)
    return AutoTokenizer.from_pretrained(path, revision=rev)

def sib(lang):
    """All 1,004 SIB-200 sentences of a FLORES language (train, dev, test), sorted by index_id."""
    rows = []
    for split in ("train", "dev", "test"):
        text = fetch(f"{SIB}{lang}/{split}.tsv").read_text(encoding="utf-8")
        rows += [(int(r["index_id"]), r["text"]) for r in csv.DictReader(io.StringIO(text), delimiter="\t")]
    return [t for _, t in sorted(rows)]

TRANSLIT = {"kaz_Cyrl": ("kaz", "Cyrl"), "kir_Cyrl": ("kir", "Cyrl"), "tat_Cyrl": ("tat", "Cyrl"),
            "bak_Cyrl": ("bak", "Cyrl"), "uzn_Latn": ("uzb", "Latn"), "azj_Latn": ("aze", "Latn"), "tur_Latn": ("tur", "Latn")}
_T = {}
def to_cts(lang, text):
    """Common Turkic Alphabet version of a sentence with TurkicNLP's transliterator."""
    if lang not in _T:
        from turkicnlp.scripts import Script
        from turkicnlp.scripts.transliterator import Transliterator
        code, scr = TRANSLIT[lang]
        _T[lang] = Transliterator(code, Script.CYRILLIC if scr == "Cyrl" else Script.LATIN, Script.COMMON_TURKIC)
    return _T[lang].transliterate(text)

_ENC = {}
NLLB_CODE = {"kaz_Cyrl": "kaz_Cyrl", "kir_Cyrl": "kir_Cyrl", "tat_Cyrl": "tat_Cyrl", "bak_Cyrl": "bak_Cyrl",
             "uzn_Latn": "uzn_Latn", "azj_Latn": "azj_Latn", "tur_Latn": "tur_Latn", "rus_Cyrl": "rus_Cyrl",
             "eng_Latn": "eng_Latn", "uig_Arab": "uig_Arab"}
def embed(model, texts, lang=None, batch=32):
    """L2-normalised mean-pooled sentence vectors. Paraphrase encoders truncate at their configured
    128 tokens; the NLLB encoder (as in TurkicNLP's embeddings processor) gets the language token of
    `lang` and is not truncated below 512. Cached on disk by model, language and text hash."""
    key = hashlib.sha1(json.dumps([model, lang, texts], ensure_ascii=False).encode()).hexdigest()[:20]
    path = CACHE / "ch13-vectors" / f"{key}.npy"
    if path.exists():
        return np.load(path)
    import torch
    torch.set_num_threads(THREADS)
    if model not in _ENC:
        from transformers import AutoModel, AutoTokenizer
        if model == "nllb":
            from transformers import AutoModelForSeq2SeqLM
            tok = AutoTokenizer.from_pretrained(NLLB_DIR)
            enc = AutoModelForSeq2SeqLM.from_pretrained(NLLB_DIR).get_encoder().eval()
            _ENC[model] = (tok, enc, 512)
        else:
            src, rev = model_source(model)
            _ENC[model] = (AutoTokenizer.from_pretrained(src, revision=rev), AutoModel.from_pretrained(src, revision=rev).eval(), 128)
    tok, enc, limit = _ENC[model]
    if model == "nllb":
        tok.src_lang = NLLB_CODE[lang]
    out = []
    with torch.inference_mode():
        for i in range(0, len(texts), batch):
            b = tok(texts[i:i + batch], padding=True, truncation=True, max_length=limit, return_tensors="pt")
            h = enc(input_ids=b["input_ids"], attention_mask=b["attention_mask"]).last_hidden_state
            m = b["attention_mask"].unsqueeze(-1).to(h.dtype)
            v = (h * m).sum(1) / m.sum(1).clamp(min=1)
            out.append(torch.nn.functional.normalize(v, dim=-1).numpy())
    vec = np.concatenate(out).astype(np.float32)
    path.parent.mkdir(parents=True, exist_ok=True); np.save(path, vec)
    return vec

def top1(q, c, targets):
    """Share of queries whose nearest candidate is the aligned one (exact duplicate target texts count)."""
    best = (q @ c.T).argmax(1)
    return float(np.mean([targets[b] == targets[i] for i, b in enumerate(best)]))

print("encoders:", ", ".join(f"{NAMES.get(n, n)} ({'local' if model_source(n)[1] is None else 'Hub'})" for n in list(MODELS)[:2]),
      f"+ NLLB encoder ({'local' if (NLLB_DIR / 'config.json').exists() else 'missing'})")

In [ ]:
import bm25s
CORPUS = [fetch(f"{KAZQAD}corpus/kazqad-corpus-v1.0-kk-part-{p}.jsonl.gz") for p in (1, 2, 3)]
def topics(split):
    rows = fetch(f"{KAZQAD}topics/kazqad-topics-v1.0-kk-{split}.tsv").read_text(encoding="utf-8").splitlines()
    return dict(r.split("\t", 1) for r in rows if r.strip())
def qrels(split):
    rel = collections.defaultdict(set)
    for r in fetch(f"{KAZQAD}qrels/kazqad-qrels-v1.0-{split}.tsv").read_text().splitlines():
        q, _, d, g = r.split("\t")
        if int(g) > 0:
            rel[q].add(d)
    return rel
WORD = re.compile(r"\w+")
def words(text):
    return WORD.findall(text.lower())

def passages():
    for p in CORPUS:
        with gzip.open(p, "rt", encoding="utf-8") as f:
            for line in f:
                d = json.loads(line)
                yield d["docid"], d["title"] + " " + d["text"]

# Context-free Apertium lemma for every lower-cased word type of the collection (cached).
LEMMA_FILE = CACHE / "ch13-kazqad-lemmas.json"
types = collections.Counter(w for _, t in passages() for w in words(t))
QUERY = {s: topics(s) for s in ("validation", "test")}
for s in QUERY:
    types.update(w for q in QUERY[s].values() for w in words(q))
lemma = json.loads(LEMMA_FILE.read_text()) if LEMMA_FILE.exists() else {}
if len(lemma) < len(types):
    import turkicnlp
    nlp = turkicnlp.Pipeline("kaz", processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")
    for w in types:
        if w not in lemma:
            x = nlp(w).words[0]
            if x.upos == "X":                                   # unknown: try the capitalised form (names)
                y = nlp(w.capitalize()).words[0]
                x = y if y.upos != "X" else x
            lemma[w] = [(x.lemma or w).lower(), x.upos]
    LEMMA_FILE.write_text(json.dumps(lemma, ensure_ascii=False))
glot = tokenizer("cis-lmu/glot500-base")
PIECES = {}
def subwords(ws):
    out = []
    for w in ws:
        if w not in PIECES:
            PIECES[w] = [p.lstrip("▁") or "▁" for p in glot.tokenize(w)]
        out += PIECES[w]
    return out
def terms(ws, mode):
    if mode == "surface":
        return ws
    if mode.startswith("prefix"):
        k = int(mode[6:]); return [w[:k] for w in ws]
    if mode == "lemma":
        return [lemma[w][0] for w in ws]
    return subwords(ws)
known = sum(c for w, c in types.items() if lemma[w][1] != "X") / sum(types.values())
print(f"KazQAD collection: {sum(1 for _ in passages()):,} passages, {sum(types.values()):,} word tokens, {len(types):,} types; "
      f"Apertium analyses {known:.1%} of tokens")

def run(mode):
    """BM25 (k1 = 1.2, b = 0.75) over the whole collection; top 100 passages per query. Cached."""
    path = CACHE / f"ch13-kazqad-run-{mode}.json"
    if path.exists():
        return json.loads(path.read_text())
    ids, docs, vocab = [], [], {}
    for d, t in passages():
        ids.append(d); docs.append([vocab.setdefault(x, len(vocab)) for x in terms(words(t), mode)])
    retriever = bm25s.BM25(k1=1.2, b=0.75)
    retriever.index(bm25s.tokenization.Tokenized(ids=docs, vocab=vocab), show_progress=False)
    del docs
    res = {}
    for s, qs in QUERY.items():
        for q, text in qs.items():
            qt = [vocab[x] for x in terms(words(text), mode) if x in vocab]
            if not qt:
                res[q] = []; continue
            scores = retriever.get_scores(qt)
            top = np.argpartition(-scores, 100)[:100]
            res[q] = [ids[i] for i in top[np.argsort(-scores[top])]]
    path.write_text(json.dumps(res)); return res

def metrics(ranked, rel):
    dcg = sum(1 / math.log2(i + 2) for i, d in enumerate(ranked[:10]) if d in rel)
    ideal = sum(1 / math.log2(i + 2) for i in range(min(10, len(rel))))
    rr = next((1 / (i + 1) for i, d in enumerate(ranked[:10]) if d in rel), 0.0)
    return dcg / ideal, len(rel & set(ranked[:100])) / len(rel), rr

def evaluate(runs, split):
    rel = qrels(split); qs = [q for q in QUERY[split] if q in rel]
    return qs, {m: np.array([metrics(r.get(q, []), rel[q]) for q in qs]) for m, r in runs.items()}

MODES = ["surface", "prefix4", "prefix5", "prefix6", "prefix7", "lemma", "subword"]
RUNS = {m: run(m) for m in MODES}
dev_q, dev = evaluate(RUNS, "validation")
test_q, test = evaluate(RUNS, "test")
best_k = max(["prefix4", "prefix5", "prefix6", "prefix7"], key=lambda m: dev[m][:, 0].mean())
print(f"validation: {len(dev_q)} queries with judged passages; test: {len(test_q)}; prefix length chosen on validation: {best_k}")
print(f"{'terms':<10}{'dev nDCG@10':>12}{'test nDCG@10':>13}{'R@100':>7}{'MRR@10':>8}{'p vs surface':>13}")
rng = np.random.default_rng(0)
for m in MODES:
    d = test[m][:, 0] - test["surface"][:, 0]
    flips = rng.choice([-1, 1], size=(10000, len(d)))
    p = (np.abs((flips * d).mean(1)) >= abs(d.mean())).mean() if m != "surface" else float("nan")   # paired sign-flip test, 10,000 draws
    print(f"{m:<10}{dev[m][:, 0].mean():12.3f}{test[m][:, 0].mean():13.3f}{test[m][:, 1].mean():7.3f}{test[m][:, 2].mean():8.3f}"
          f"{'' if m == 'surface' else ('      <0.0001' if p < 1e-4 else f'{p:13.4f}')}")

def paired_p(a, b, n=10000):
    d = test[a][:, 0] - test[b][:, 0]
    flips = rng.choice([-1, 1], size=(n, len(d)))
    return d.mean(), (np.abs((flips * d).mean(1)) >= abs(d.mean())).mean()
for a, b in (("lemma", best_k), ("subword", best_k), ("subword", "lemma")):
    diff, p = paired_p(a, b)
    print(f"  {a} minus {b}: test nDCG@10 {diff:+.3f}, p = {p:.3f}")

# Where do lemmas lose? Test queries by number of words the analyser changed or could not analyse.
qtext = QUERY["test"]
print("\ntest queries by analyser outcome (mean nDCG@10, lemma minus surface):")
groups = collections.defaultdict(list)
for i, q in enumerate(test_q):
    ws = words(qtext[q]); unk = sum(lemma[w][1] == "X" for w in ws)
    groups["has unanalysed word" if unk else "all words analysed"].append(test["lemma"][i, 0] - test["surface"][i, 0])
for g, v in groups.items():
    print(f"  {g:<22} {len(v):>5} queries  {np.mean(v):+.3f}")
loss = sorted(range(len(test_q)), key=lambda i: test["lemma"][i, 0] - test["surface"][i, 0])[:8]
print("largest losses from lemmas:")
for i in loss:
    q = test_q[i]; ws = words(qtext[q])
    print(f"  {test['surface'][i, 0]:.2f}->{test['lemma'][i, 0]:.2f}  " + " ".join(f"{w}>{lemma[w][0]}" if lemma[w][0] != w else w for w in ws))

## Worked solution

Output of `experiments/ch13_exercises/run_ch13_exercises.py`:

```
KazQAD collection: 825,309 passages, 31,226,421 word tokens, 1,241,260 types; Apertium analyses 82.1% of tokens
validation: 548 queries with judged passages; test: 1929; prefix length chosen on validation: prefix5
terms      dev nDCG@10 test nDCG@10  R@100  MRR@10 p vs surface
surface          0.140        0.140  0.445   0.130
prefix4          0.197        0.193  0.602   0.175      <0.0001
prefix5          0.200        0.198  0.608   0.180      <0.0001
prefix6          0.195        0.186  0.574   0.172      <0.0001
prefix7          0.181        0.172  0.556   0.158      <0.0001
lemma            0.202        0.200  0.602   0.182      <0.0001
subword          0.222        0.204  0.619   0.182      <0.0001
  lemma minus prefix5: test nDCG@10 +0.002, p = 0.634
  subword minus prefix5: test nDCG@10 +0.006, p = 0.179
  subword minus lemma: test nDCG@10 +0.004, p = 0.382

test queries by analyser outcome (mean nDCG@10, lemma minus surface):
  all words analysed      1157 queries  +0.069
  has unanalysed word      772 queries  +0.048
largest losses from lemmas:
  1.00->0.00  алғашқы дүние жүзін>жүз айналып>айнал жүзіп>жүз өтуге>өт кеткен>кет уақыт қай жылдарда>жыл
  1.00->0.00  б майлин шығармаларындағы>шығарма типтік образы>образ кім
  1.00->0.00  шығыстың>шық аристотелі>аристотель атанған>атан адам кім
  1.00->0.33  гомозиготалы генотип қандай әріппен>әріп белгіленеді>белгілен
  1.00->0.36  ресей бодандығын 1817 жылы бірінші болып>бол қабылдаған>қабылда ұлы жүздің>жүз руы>ру
  1.00->0.36  1836 1838 жылдары>жыл бөкей ордасында>орда болған>бол көтерілістің>көтеріл басты>бас қозғаушы>қозға күші>күш
  0.63->0.00  түрді>түр латын тілінде>тіл қос атпен>ат атауды>ата ғылымға>ғылым ұсынған>ұсын ғалым кім
  0.63->0.00  тамақтан уланудың>улану себептері>себеп қандай
```

**Reduction methods.** Surface BM25 scores 0.140 nDCG@10 on 1,929 test questions. The validation-selected five-letter prefix scores 0.198, lemmas 0.200 and subwords 0.204. Recorded paired sign-flip tests find improvements over surface forms, but not differences among these three alternatives. A failure to detect a difference does not establish equivalence.

**Lemma errors to inspect.** Apertium analyses 82.1% of collection word tokens; unanalysed words retain their surface forms. Some queries with large losses contain ambiguous or merged lemma strings, such as *жүзін* and *жүзіп* mapping to *жүз*. Inspect the intended readings and retrieved passages before attributing a rank change to one lemma. These examples do not quantify how many losses are caused by analysis errors.

**Scope.** The lemma map selects readings without sentence context. The exam questions and available relevance judgements define this experiment; unjudged relevant passages may exist. Compare a context-sensitive analyser and hybrid retriever on the same frozen splits rather than inferring their scores from these results.

## Your turn — extension

1. Use the most frequent reading in context (TurkicNLP's disambiguated `morph` output on whole passages) instead of context-free lemmas, on a sample of the collection.
2. Combine prefix-5 and subword BM25 scores (sum of normalised scores) and tune the weight on validation.